# Task 1 Text preprocessing using the Coachella dataset

This notebook extracts hashtags and email addresses from the original tweets, then applies all seven required cleaning functions to a new `new_tweets_text` column. Original columns and row order are preserved.

The supplied CSV is not valid UTF-8 or Windows-1252. Latin-1 preserves its bytes when loading; the required ASCII cleaning removes non-ASCII characters from the new text. Existing corrupted characters cannot be reliably reconstructed.

Install dependencies if needed: `pip install nltk pandas`. The code downloads only NLTK's English stopword corpus if it is missing.

## Extraction and cleaning functions

Hashtags retain their leading `#`, original case, order and duplicates. Email extraction uses a practical pattern for ordinary addresses. Both extracted columns are saved as JSON lists (`[]` means no match).

The cleaning pipeline follows the assignment order: usernames, HTTP/HTTPS links, non-ASCII characters, lowercase conversion, English stopwords, digits and special characters. A final stopword pass removes fragments exposed by punctuation removal. Hashtag words remain in the cleaned text after removing `#` and digits. Emails are extracted separately; their remaining alphabetic text is cleaned under the same rules. No stemming, lemmatization or spelling correction is added because the assignment does not request them.

In [1]:
"""Task 1: extract metadata and preprocess the Coachella tweets."""
import argparse
import csv
import json
import re
from pathlib import Path
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer

# Download only the corpus used by this assignment, if absent.
try:
    STOP_WORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True, raise_on_error=True)
    STOP_WORDS = set(stopwords.words('english'))
TOKENIZER = TweetTokenizer()
HASHTAG_PATTERN = re.compile(r'(?<!\w)#[\w]+', re.UNICODE)
EMAIL_PATTERN = re.compile(r"(?<![\w.+-])[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?(?:\.[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?)+")


def extract_hashtags(text):
    """Retain the #, spelling, order and repeated occurrences."""
    return HASHTAG_PATTERN.findall(text)


def extract_emails(text):
    return EMAIL_PATTERN.findall(text)


def remove_usernames(text):
    """Remove mentions without treating an email domain as a mention."""
    return re.sub(r'(?<![\w@.+-])@[A-Za-z0-9_]+', ' ', text)


def remove_links(text):
    return re.sub(r'https?://\S+', ' ', text, flags=re.IGNORECASE)


def remove_non_ascii_symbols(text):
    return text.encode('ascii', errors='ignore').decode('ascii')


def to_lower(text):
    return text.lower()


def remove_stop_words(text):
    """TweetTokenizer needs no Punkt download and handles punctuation."""
    return ' '.join(token for token in TOKENIZER.tokenize(text)
                    if token.lower() not in STOP_WORDS)


def remove_digits(text):
    return re.sub(r'\d+', '', text)


def remove_special_characters(text):
    # Use spaces so punctuation-separated words do not become one word.
    return ' '.join(re.sub(r'[^a-zA-Z\s]', ' ', text).split())


def preprocess_text(text):
    text = '' if text is None else str(text)
    for function in (remove_usernames, remove_links, remove_non_ascii_symbols,
                     to_lower, remove_stop_words, remove_digits,
                     remove_special_characters):
        text = function(text)
    # Punctuation removal can expose stopwords (e.g. parts of contractions).
    return remove_stop_words(text)


def process_dataset(input_path, output_path):
    # This supplied file is not valid UTF-8 or Windows-1252. Latin-1 is
    # a lossless one-byte mapping; existing mojibake is removed by ASCII cleaning.
    with open(input_path, encoding='latin-1', newline='') as source:
        reader = csv.DictReader(source)
        columns = reader.fieldnames
        if not columns or 'text' not in columns:
            raise ValueError("The CSV must contain a 'text' column.")
        rows = list(reader)
    for row in rows:
        original = row['text'] or ''
        row['hashtags'] = json.dumps(extract_hashtags(original), ensure_ascii=False)
        row['emails'] = json.dumps(extract_emails(original), ensure_ascii=False)
        row['new_tweets_text'] = preprocess_text(original)
    with open(output_path, 'w', encoding='utf-8', newline='') as destination:
        writer = csv.DictWriter(destination, fieldnames=columns +
                                ['hashtags', 'emails', 'new_tweets_text'])
        writer.writeheader()
        writer.writerows(rows)
    return rows


def summarize(rows):
    return {
        'tweets': len(rows),
        'tweets_with_hashtags': sum(bool(json.loads(r['hashtags'])) for r in rows),
        'hashtag_occurrences': sum(len(json.loads(r['hashtags'])) for r in rows),
        'tweets_with_emails': sum(bool(json.loads(r['emails'])) for r in rows),
        'email_occurrences': sum(len(json.loads(r['emails'])) for r in rows),
        'empty_cleaned_tweets': sum(not r['new_tweets_text'] for r in rows),
    }




## Process the complete dataset

In [2]:
rows = process_dataset('Coachella-2015-2-DFE.csv', 'coachella_processed.csv')
print(json.dumps(summarize(rows), indent=2))

{
  "tweets": 3846,
  "tweets_with_hashtags": 3832,
  "hashtag_occurrences": 5475,
  "tweets_with_emails": 0,
  "email_occurrences": 0,
  "empty_cleaned_tweets": 0
}


## Example results

In [3]:
import pandas as pd
pd.set_option('display.max_colwidth', 150)
pd.DataFrame(rows)[['text', 'hashtags', 'emails', 'new_tweets_text']].head(5)

,text,hashtags,emails,new_tweets_text
0,#Coachella2015 tickets selling out in less than 40 minutes _Ù_¦_Ù___Ù___Ù÷_ÙÎµ_ÙÎµ_Ù___Ù_¦ http://t.co/SmoXyteIMJ,"[""#Coachella2015""]",[],coachella tickets selling less minutes
1,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ÙÓ´_ÙÓ´_ÙÓ´ #Coachella2015 http://t.co/Ov2eCJtAvR,"[""#Coachella2015""]",[],rt wait absolute fire coachella
2,#Coachella2015 #VIP passes secured! See you there bitchesssss,"[""#Coachella2015"", ""#VIP""]",[],coachella vip passes secured see bitchesssss
3,PhillyÛªs @warondrugsjams will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon: http://t.co/wzaAE5wYNT http://t.co/sFL9I5sD20,"[""#Coachella2015"", ""#GovBall2015""]",[],phillys play coachella govball watch jimmy fallon
4,If briana and her mom out to #Coachella2015 im out with them !!! _Ù÷Ý_Ù÷Ý_Ù÷Ý_ÙÕÄ,"[""#Coachella2015""]",[],briana mom coachella im


## Check extraction and cleaning
The supplied dataset may have no emails. A separate illustrative tweet checks email extraction, mentions, links, accents, numbers and punctuation.

In [4]:
example = "Hey @beyonce! The #Coachella2015 show is AMAZING! Email example@gmail.com or user@yahoo.com. https://example.com Beyoncé 😊 100"
print('Hashtags:', extract_hashtags(example))
print('Emails:', extract_emails(example))
print('Cleaned:', preprocess_text(example))
assert extract_hashtags(example) == ['#Coachella2015']
assert extract_emails(example) == ['example@gmail.com', 'user@yahoo.com']
assert '@beyonce' not in remove_usernames(example)
assert 'example@gmail.com' in remove_usernames(example)
assert 'https://' not in remove_links(example)
assert remove_non_ascii_symbols('Beyoncé 😊') == 'Beyonc '
assert to_lower('COACHELLA') == 'coachella'
assert remove_stop_words('the music and a festival') == 'music festival'
assert remove_digits('2015 music 100') == ' music '
assert remove_special_characters('music/festival!') == 'music festival'
assert preprocess_text(None) == ''
assert len(rows) == 3846
for row in rows:
    cleaned = row['new_tweets_text']
    assert re.fullmatch(r'[a-z ]*', cleaned)
    assert not (set(cleaned.split()) & STOP_WORDS)
    assert cleaned == ' '.join(cleaned.split())
    assert json.loads(row['hashtags']) == extract_hashtags(row['text'])
    assert json.loads(row['emails']) == extract_emails(row['text'])
with open('coachella_processed.csv', encoding='utf-8', newline='') as f:
    saved = list(csv.DictReader(f))
assert saved == rows
print('All checks passed; CSV round-trip preserved all rows and values.')

Hashtags: ['#Coachella2015']
Emails: ['example@gmail.com', 'user@yahoo.com']
Cleaned: hey coachella show amazing email example gmail com user yahoo com beyonc
All checks passed; CSV round-trip preserved all rows and values.


## Interpretation

The output CSV contains the original 10 columns plus `hashtags`, `emails` and `new_tweets_text`. The printed summary reports the measured extraction counts and empty cleaned tweets. Removing stopwords and punctuation simplifies the vocabulary, but also removes negation and expressive cues that could matter for later sentiment analysis.